# 🏷️ Encoding Categorical Variables
**One-line description:** Convert text categories into numbers that machine learning models can actually understand.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadtalhaishtiaq/ai-orchestrator/blob/main/01-data-preprocessing/02_encoding_categorical.ipynb)


In [ ]:
# Install required libraries (run this cell first in Google Colab)
!pip install scikit-learn pandas numpy matplotlib seaborn category_encoders --quiet


## 📖 What is Categorical Encoding?

Categorical encoding is the process of converting **text-based or discrete categories** into **numeric representations** that ML algorithms can process.

**Analogy:** Imagine teaching a child to sort fruits — if you say "apple=1, banana=2, cherry=3", they now understand the ordering mathematically. But does banana really come "between" apple and cherry? That's the core challenge of encoding!

### Types of Categorical Variables:
| Type | Description | Example |
|------|-------------|---------|
| **Nominal** | No inherent order | City: Paris, London, Tokyo |
| **Ordinal** | Has natural order | Size: Small < Medium < Large |
| **Binary** | Only 2 categories | Gender: Male/Female |
| **High Cardinality** | Many unique values | ZIP codes, User IDs |


## 💡 Why Does It Matter?

- ML models work with **numbers**, not strings
- Wrong encoding introduces **false ordering** (label encoding for nominal data)
- **One-hot encoding** on high-cardinality features causes dimensionality explosion
- Choosing the right encoding can significantly **improve model accuracy**


## ⚙️ How Does It Work?

We'll cover 6 encoding strategies:
1. **Label Encoding** — each category gets a unique integer
2. **Ordinal Encoding** — integers respect the natural order
3. **One-Hot Encoding (OHE)** — binary columns for each category
4. **Target Encoding** — replace category with mean of target variable
5. **Frequency Encoding** — replace category with its frequency count
6. **Binary Encoding** — compact representation using binary digits


## 🛠️ Hands-on Code

### Step 1: Create an Employee/HR synthetic dataset


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, OneHotEncoder
import category_encoders as ce
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
n = 1000

# Ordinal features (have a natural order)
education_levels = ['High School', 'Associate', 'Bachelor', 'Master', 'PhD']
job_levels = ['Junior', 'Mid', 'Senior', 'Lead', 'Principal']
performance = ['Poor', 'Below Average', 'Average', 'Good', 'Excellent']

# Nominal features (no natural order)
departments = ['Engineering', 'Marketing', 'Sales', 'HR', 'Finance', 'Operations', 'Legal']
cities = ['New York', 'San Francisco', 'Chicago', 'Austin', 'Boston', 'Seattle', 'Denver']
employment_type = ['Full-time', 'Part-time', 'Contract', 'Intern']

df = pd.DataFrame({
    'EmployeeID':       range(1001, 1001 + n),
    'Department':       np.random.choice(departments, n),
    'City':             np.random.choice(cities, n),
    'EmploymentType':   np.random.choice(employment_type, n, p=[0.7, 0.1, 0.15, 0.05]),
    'EducationLevel':   np.random.choice(education_levels, n, p=[0.1, 0.1, 0.45, 0.28, 0.07]),
    'JobLevel':         np.random.choice(job_levels, n, p=[0.25, 0.30, 0.25, 0.12, 0.08]),
    'PerformanceRating': np.random.choice(performance, n, p=[0.05, 0.15, 0.40, 0.30, 0.10]),
    'YearsExperience':  np.random.randint(0, 25, n),
    'Salary':           np.random.normal(75000, 25000, n).astype(int).clip(30000, 200000),
    'Attrition':        np.random.choice([0, 1], n, p=[0.84, 0.16]),  # Target variable
})

print("Dataset shape:", df.shape)
print("\nDataset info:")
print(df.dtypes)
print("\nFirst 5 rows:")
df.head()


In [ ]:
# --- Visualization 1: Distribution of categorical features ---
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

cat_cols = ['Department', 'City', 'EmploymentType', 'EducationLevel', 'JobLevel', 'PerformanceRating']
colors = plt.cm.Set2.colors

for ax, col in zip(axes.flat, cat_cols):
    counts = df[col].value_counts()
    bars = ax.bar(range(len(counts)), counts.values, color=colors[:len(counts)], edgecolor='black')
    ax.set_xticks(range(len(counts)))
    ax.set_xticklabels(counts.index, rotation=45, ha='right', fontsize=9)
    ax.set_title(f'{col} Distribution', fontsize=11, fontweight='bold')
    ax.set_ylabel('Count')
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(val), ha='center', fontsize=8)

plt.suptitle('Categorical Feature Distributions in HR Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('categorical_distributions.png', dpi=100, bbox_inches='tight')
plt.show()


### Step 2: Label Encoding (for ordinal or binary features)


In [ ]:
# Label Encoding: assigns integer 0 to N-1 for each unique category
# CAUTION: implies order — only use for truly ordinal features or binary features!

le = LabelEncoder()
df_encoded = df.copy()

# Binary feature — safe to use label encoding
df_encoded['EmploymentType_LE'] = le.fit_transform(df['EmploymentType'])
print("Label Encoding — EmploymentType:")
mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(f"  Mapping: {mapping}")
print()

# Department — WRONG use of label encoding (nominal feature)
df_encoded['Department_LE'] = le.fit_transform(df['Department'])
print("Label Encoding — Department (BAD for nominal!):")
mapping_dept = dict(zip(le.classes_, le.transform(le.classes_)))
print(f"  Mapping: {mapping_dept}")
print("  ⚠️  This implies Engineering < Finance < HR which is meaningless!")


In [ ]:
# --- Ordinal Encoding: respects the natural order ---
# Perfect for: EducationLevel, JobLevel, PerformanceRating

education_order = [['High School', 'Associate', 'Bachelor', 'Master', 'PhD']]
job_order = [['Junior', 'Mid', 'Senior', 'Lead', 'Principal']]
perf_order = [['Poor', 'Below Average', 'Average', 'Good', 'Excellent']]

oe_edu = OrdinalEncoder(categories=education_order)
oe_job = OrdinalEncoder(categories=job_order)
oe_perf = OrdinalEncoder(categories=perf_order)

df_encoded['EducationLevel_OE'] = oe_edu.fit_transform(df[['EducationLevel']])
df_encoded['JobLevel_OE'] = oe_job.fit_transform(df[['JobLevel']])
df_encoded['PerformanceRating_OE'] = oe_perf.fit_transform(df[['PerformanceRating']])

print("Ordinal Encoding — EducationLevel (order preserved):")
for level, code_val in zip(education_order[0], range(5)):
    print(f"  {level} → {code_val:.1f}")


### Step 3: One-Hot Encoding (for nominal features with low cardinality)


In [ ]:
# One-Hot Encoding: creates a binary column for each category
# Good for: nominal features with < ~15 unique values
# Bad for: high-cardinality features (creates too many columns)

# Using pandas get_dummies
dept_ohe_pandas = pd.get_dummies(df['Department'], prefix='Dept', drop_first=False)
print("pandas get_dummies result:")
print(dept_ohe_pandas.head(3))
print(f"Shape: {dept_ohe_pandas.shape} — {dept_ohe_pandas.shape[1]} new columns created")

print()

# Using sklearn OneHotEncoder
ohe = OneHotEncoder(drop='first', sparse_output=False)  # drop='first' avoids dummy variable trap
dept_ohe_sklearn = pd.DataFrame(
    ohe.fit_transform(df[['Department']]),
    columns=ohe.get_feature_names_out(['Department'])
)
print("sklearn OneHotEncoder result (drop='first'):")
print(dept_ohe_sklearn.head(3))
print(f"Shape: {dept_ohe_sklearn.shape} — drops one column to avoid multicollinearity")


### Step 4: Target, Frequency, and Binary Encoding


In [ ]:
# --- Target Encoding: replace category with mean of target variable ---
# Useful for high-cardinality features, but prone to target leakage!
# Always use cross-validation or out-of-fold encoding in practice.

te = ce.TargetEncoder(cols=['Department', 'City'])
df_target_enc = te.fit_transform(df[['Department', 'City']], df['Salary'])
print("Target Encoding (Department → mean Salary):")
dept_means = df_target_enc['Department'].round(0)
print(df[['Department']].join(dept_means.rename('Dept_TargetEnc')).drop_duplicates().sort_values('Dept_TargetEnc').to_string())

print()
# --- Frequency Encoding: replace category with its count/frequency ---
freq_map = df['Department'].value_counts().to_dict()
df_encoded['Department_FreqEnc'] = df['Department'].map(freq_map)
print("Frequency Encoding (Department → count):")
print(df[['Department']].assign(FreqEnc=df_encoded['Department_FreqEnc']).drop_duplicates().sort_values('FreqEnc'))


In [ ]:
# --- Visualization 2: Impact of encoding choices ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Salary vs Department (using target encoding values)
dept_salary = df.groupby('Department')['Salary'].mean().sort_values()
axes[0].barh(dept_salary.index, dept_salary.values, color='steelblue', edgecolor='black')
axes[0].set_title('Mean Salary by Department
(Target Encoding approximates this)', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Mean Salary ($)')

# OHE column count growth with cardinality
unique_counts = [df[c].nunique() for c in cat_cols]
ohe_col_counts = unique_counts  # OHE creates n columns (or n-1 with drop)
axes[1].bar(cat_cols, unique_counts, color='coral', edgecolor='black')
axes[1].set_title('Unique Values per Feature
(= OHE columns created)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Number of unique values')
axes[1].set_xticklabels(cat_cols, rotation=45, ha='right', fontsize=9)

# Ordinal vs Label encoding comparison for JobLevel
job_map_ordinal = {'Junior': 0, 'Mid': 1, 'Senior': 2, 'Lead': 3, 'Principal': 4}
job_salary = df.groupby('JobLevel')['Salary'].mean()
job_salary.index = [job_map_ordinal[j] for j in job_salary.index]
job_salary = job_salary.sort_index()
axes[2].plot(job_salary.index, job_salary.values, 'o-', color='green', linewidth=2, markersize=8)
axes[2].set_title('Salary Increases Monotonically
with Ordinal JobLevel Encoding', fontsize=11, fontweight='bold')
axes[2].set_xlabel('Ordinal Encoding (0=Junior, 4=Principal)')
axes[2].set_ylabel('Mean Salary ($)')
axes[2].set_xticks(range(5))
axes[2].set_xticklabels(['Junior','Mid','Senior','Lead','Principal'], rotation=30)

plt.suptitle('Encoding Strategy Impact Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('encoding_comparison.png', dpi=100, bbox_inches='tight')
plt.show()


## 📌 Key Takeaways

- **Label Encoding** only for ordinal or binary features — never for nominal with 3+ categories
- **Ordinal Encoding** when natural order exists — always specify the category order explicitly
- **One-Hot Encoding** for nominal features with low cardinality (< ~15 unique values)
- **Target Encoding** for high-cardinality nominal features — but use cross-validation to prevent leakage
- **Frequency Encoding** is a simple, leakage-free alternative to target encoding
- Use `drop='first'` in OHE to avoid the **dummy variable trap** (multicollinearity)
- Always fit encoders on **training data only**, then transform both train and test sets
